# Count parameters for `symmetric_47_G.pth`

Notebook nay load checkpoint va dem so parameter/tensor elements trong file:

`/Users/mac/Documents/DATN/source/DocNLC/ckpts/symmetric_47_G.pth`

In [13]:
from pathlib import Path
from collections import OrderedDict, defaultdict

import torch

CKPT_PATH = Path('/Users/mac/Documents/DATN/source/DocNLC/ckpts/26_G.pth')
CKPT_PATH.exists(), CKPT_PATH

(True, PosixPath('/Users/mac/Documents/DATN/source/DocNLC/ckpts/26_G.pth'))

In [14]:
def is_tensor_dict(obj):
    return isinstance(obj, dict) and len(obj) > 0 and all(torch.is_tensor(v) for v in obj.values())


def clean_module_prefix(state_dict):
    cleaned = OrderedDict()
    for key, value in state_dict.items():
        cleaned[key[7:] if key.startswith('module.') else key] = value
    return cleaned


def extract_state_dict(checkpoint):
    if is_tensor_dict(checkpoint):
        return clean_module_prefix(checkpoint)

    if isinstance(checkpoint, dict):
        for key in ('state_dict', 'model', 'netG', 'params', 'params_ema'):
            value = checkpoint.get(key)
            if is_tensor_dict(value):
                return clean_module_prefix(value)

        tensor_items = OrderedDict((k, v) for k, v in checkpoint.items() if torch.is_tensor(v))
        if tensor_items:
            return clean_module_prefix(tensor_items)

    raise TypeError(f'Cannot find a tensor state_dict in checkpoint type: {type(checkpoint)}')


checkpoint = torch.load(CKPT_PATH, map_location='cpu')
state_dict = extract_state_dict(checkpoint)

print('Checkpoint type:', type(checkpoint))
print('Number of tensor entries:', len(state_dict))
print('First 10 keys:')
for key in list(state_dict.keys())[:10]:
    print(' ', key, tuple(state_dict[key].shape))

Checkpoint type: <class 'collections.OrderedDict'>
Number of tensor entries: 56
First 10 keys:
  conv1_1.conv1.weight (32, 3, 1, 1)
  conv1_1.conv2_1.weight (32, 32, 3, 3)
  conv1_1.conv2_2.weight (32, 32, 3, 3)
  conv1_1.instance.weight (32,)
  conv1_1.instance.bias (32,)
  conv1_1.interative.0.weight (32, 64, 3, 3)
  conv1_1.process.0.weight (16, 64, 3, 3)
  conv1_1.process.0.bias (16,)
  conv1_1.process.2.weight (64, 16, 3, 3)
  conv1_1.process.2.bias (64,)


In [15]:
total_params = sum(t.numel() for t in state_dict.values())
total_size_bytes = sum(t.numel() * t.element_size() for t in state_dict.values())

print(f'Checkpoint: {CKPT_PATH}')
print(f'Total parameters / tensor elements: {total_params:,}')
print(f'Total parameters in millions: {total_params / 1_000_000:.4f} M')
print(f'Raw tensor size: {total_size_bytes / (1024 ** 2):.2f} MiB')

Checkpoint: /Users/mac/Documents/DATN/source/DocNLC/ckpts/26_G.pth
Total parameters / tensor elements: 7,816,883
Total parameters in millions: 7.8169 M
Raw tensor size: 29.82 MiB


In [16]:
groups = defaultdict(int)
for name, tensor in state_dict.items():
    group = name.split('.')[0]
    groups[group] += tensor.numel()

print('Parameter count by top-level module:')
for group, count in sorted(groups.items(), key=lambda item: item[1], reverse=True):
    print(f'{group:<20} {count:>15,}  ({count / total_params * 100:6.2f}%)')

Parameter count by top-level module:
conv5_2                    2,359,808  ( 30.19%)
conv5_1                    1,180,160  ( 15.10%)
conv6_1                    1,179,904  ( 15.09%)
conv4_2                      590,080  (  7.55%)
conv6_2                      590,080  (  7.55%)
upv6                         524,544  (  6.71%)
conv4_1                      295,168  (  3.78%)
conv7_1                      295,040  (  3.77%)
conv3_2                      147,584  (  1.89%)
conv7_2                      147,584  (  1.89%)
upv7                         131,200  (  1.68%)
conv3_1                       73,856  (  0.94%)
conv8_1                       73,792  (  0.94%)
conv1_1                       57,616  (  0.74%)
conv2_2                       36,928  (  0.47%)
conv8_2                       36,928  (  0.47%)
upv8                          32,832  (  0.42%)
conv2_1                       18,496  (  0.24%)
conv9_1                       18,464  (  0.24%)
conv1_2                        9,248  (  0.12%)
con

In [17]:
rows = []
for name, tensor in state_dict.items():
    rows.append({
        'name': name,
        'shape': tuple(tensor.shape),
        'num_params': tensor.numel(),
        'dtype': str(tensor.dtype),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows).sort_values('num_params', ascending=False).reset_index(drop=True)
    display(df)
except ImportError:
    for row in sorted(rows, key=lambda item: item['num_params'], reverse=True):
        print(f"{row['name']:<35} {str(row['shape']):<24} {row['num_params']:>12,} {row['dtype']}")

,name,shape,num_params,dtype
0,conv5_2.weight,"(512, 512, 3, 3)",2359296,torch.float32
1,conv6_1.weight,"(256, 512, 3, 3)",1179648,torch.float32
2,conv5_1.weight,"(512, 256, 3, 3)",1179648,torch.float32
3,conv6_2.weight,"(256, 256, 3, 3)",589824,torch.float32
4,conv4_2.weight,"(256, 256, 3, 3)",589824,torch.float32
5,upv6.weight,"(512, 256, 2, 2)",524288,torch.float32
6,conv7_1.weight,"(128, 256, 3, 3)",294912,torch.float32
7,conv4_1.weight,"(256, 128, 3, 3)",294912,torch.float32
8,conv7_2.weight,"(128, 128, 3, 3)",147456,torch.float32
9,conv3_2.weight,"(128, 128, 3, 3)",147456,torch.float32


## Optional: compare with the `SeeInDark` architecture

Cell duoi day thu khoi tao model trong `models/multitask_Barlow_model_new.py` va load checkpoint de so sanh. Neu moi truong notebook thieu dependency nhu `cv2`/`numpy`, co the bo qua cell nay; cac cell ben tren van dem checkpoint truc tiep.

In [18]:
try:
    from models.multitask_Barlow_model_new import SeeInDark

    model = SeeInDark()
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    model_total = sum(p.numel() for p in model.parameters())
    model_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'Model total parameters: {model_total:,}')
    print(f'Model trainable parameters: {model_trainable:,}')
    print(f'Missing keys: {len(missing)}')
    print(f'Unexpected keys: {len(unexpected)}')
    if missing:
        print('Missing:', missing[:20])
    if unexpected:
        print('Unexpected:', unexpected[:20])
except Exception as exc:
    print('Skip architecture comparison because import/load failed:')
    print(type(exc).__name__, exc)

Model total parameters: 7,816,883
Model trainable parameters: 7,816,883
Missing keys: 0
Unexpected keys: 0
